# 04 — LDM: Latent Diffusion Models

**Paper:** *High-Resolution Image Synthesis with Latent Diffusion Models* (Rombach et al., CVPR 2022)  
**arXiv:** https://arxiv.org/abs/2112.10752  
**Also known as:** Stable Diffusion

---

## The Problem: Diffusion in Pixel Space is Expensive

DDPM/IDDPM/ADM all operate **directly on pixels**.  
For a 512×512 image, that's 786,432 dimensions.  
Training and inference require hundreds of function evaluations in this space — extremely slow.

**LDM's insight:** First compress images into a low-dimensional latent space using a VAE.  
Then run diffusion in that **latent space** — much cheaper!

## Two-Stage Design

![LDM Architecture](./figures/latent-diffusion-arch.png)

### Stage 1: Perceptual Compression (VAE)

```
Image x  (H, W, 3)
    │  Encoder E
    ▼
Latent z  (H/f, W/f, C)    ← f = downsampling factor (f=8 in SD)
    │  Decoder D
    ▼
Reconstructed x̂  (H, W, 3)
```

Train with: L = L_reconstruction + λ_KL · KL + λ_perceptual · L_LPIPS + λ_GAN · L_patch_GAN

### Stage 2: Latent Diffusion

```
z ~ E(x)                         ← encode real image to latent
z_T ~ N(0, I)                    ← start from noise at inference
ε_θ(z_t, t, τ_θ(y))             ← U-Net denoises in latent space
                                      conditioned on text/image/class via cross-attention
D(z_0)                           ← decode to pixel space
```

## Conditioning via Cross-Attention

LDM unifies all conditioning modalities (text, class, image) through **cross-attention** in the U-Net:

```python
# Inside each U-Net transformer block:
Q = W_Q(z_t_features)     # from latent
K = W_K(τ(y))             # from condition encoder τ
V = W_V(τ(y))
z_attended = softmax(QK^T / √d) · V
```

Condition encoders `τ_θ(y)`:
- **Text** → CLIP text encoder or BERT
- **Class** → learnable embedding
- **Image** → CLIP image encoder or another CNN

## Perceptual Compression: Why KL + Perceptual + GAN?

![Perceptual Compression](./figures/ldm_perceptual.png)

| Loss | Purpose |
|------|---------|
| **L_reconstruction** | Pixel-level accuracy (MSE) |
| **L_LPIPS** | Perceptual similarity (VGG features) |
| **L_KL** | Regularise latent space towards N(0,I) |
| **L_patch_GAN** | Sharpness — penalise blurry reconstructions |

The KL term is weighted very low (λ_KL = 1e-6) to avoid over-compressing.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math

# ── VAE Encoder/Decoder (simplified LDM VQ-f8 style) ──

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.GroupNorm(8, ch), nn.SiLU(),
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.GroupNorm(8, ch), nn.SiLU(),
            nn.Conv2d(ch, ch, 3, padding=1)
        )
    def forward(self, x): return x + self.net(x)


class LDMEncoder(nn.Module):
    """Compress (3, H, W) → (4, H/8, W/8) via 3 downsampling stages"""
    def __init__(self, in_ch=3, latent_ch=4, base_ch=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, base_ch, 3, padding=1),
            ResBlock(base_ch),
            nn.Conv2d(base_ch, base_ch*2, 4, stride=2, padding=1),   # /2
            ResBlock(base_ch*2),
            nn.Conv2d(base_ch*2, base_ch*4, 4, stride=2, padding=1), # /4
            ResBlock(base_ch*4),
            nn.Conv2d(base_ch*4, base_ch*8, 4, stride=2, padding=1), # /8
            ResBlock(base_ch*8),
            nn.GroupNorm(8, base_ch*8), nn.SiLU(),
            nn.Conv2d(base_ch*8, latent_ch*2, 1)  # *2 for mean + logvar
        )
    
    def forward(self, x):
        h = self.net(x)
        mean, logvar = h.chunk(2, dim=1)
        logvar = logvar.clamp(-30, 20)
        std = (0.5 * logvar).exp()
        z = mean + std * torch.randn_like(std)   # reparameterization
        kl = -0.5 * (1 + logvar - mean**2 - logvar.exp()).mean()
        return z, kl


class LDMDecoder(nn.Module):
    """Expand (4, H/8, W/8) → (3, H, W) via 3 upsampling stages"""
    def __init__(self, latent_ch=4, out_ch=3, base_ch=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(latent_ch, base_ch*8, 3, padding=1),
            ResBlock(base_ch*8),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(base_ch*8, base_ch*4, 3, padding=1), # ×2
            ResBlock(base_ch*4),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(base_ch*4, base_ch*2, 3, padding=1), # ×4
            ResBlock(base_ch*2),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(base_ch*2, base_ch, 3, padding=1),   # ×8
            ResBlock(base_ch),
            nn.GroupNorm(8, base_ch), nn.SiLU(),
            nn.Conv2d(base_ch, out_ch, 3, padding=1),
            nn.Tanh()
        )
    def forward(self, z): return self.net(z)


# Test VAE
encoder = LDMEncoder()
decoder = LDMDecoder()
x = torch.randn(2, 3, 64, 64)         # (B, 3, 64, 64) pixel image
z, kl = encoder(x)                     # (B, 4, 8, 8) latent
x_hat = decoder(z)                     # (B, 3, 64, 64) reconstruction
print(f"Image: {x.shape}  →  Latent: {z.shape}  →  Recon: {x_hat.shape}")
print(f"Compression ratio: {x.numel() / z.numel():.1f}×")
print(f"KL: {kl.item():.4f}")

In [ ]:
# ── Cross-Attention Conditioning Block ──

class CrossAttention(nn.Module):
    """
    Cross-attention between latent features (query) and condition (key/value).
    Used inside LDM's U-Net transformer blocks.
    """
    def __init__(self, d_model=64, d_cond=128, n_heads=4):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.scale   = (d_model // n_heads) ** -0.5
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_cond,  d_model, bias=False)
        self.v = nn.Linear(d_cond,  d_model, bias=False)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, context):
        # x:       (B, N, d_model)   — flattened spatial features
        # context: (B, L, d_cond)    — text/class/image tokens
        B, N, _ = x.shape
        H = self.n_heads
        D = x.shape[-1] // H

        Q = self.q(x).reshape(B, N, H, D).transpose(1, 2)
        K = self.k(context).reshape(B, -1, H, D).transpose(1, 2)
        V = self.v(context).reshape(B, -1, H, D).transpose(1, 2)

        attn = (Q @ K.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out  = (attn @ V).transpose(1, 2).reshape(B, N, -1)
        return self.out(out)


# Demo: condition latent features on text tokens
cross_attn = CrossAttention(d_model=64, d_cond=128, n_heads=4)
latent_feats = torch.randn(2, 64, 64)   # (B, spatial_tokens, d_model)
text_tokens   = torch.randn(2, 77, 128) # (B, text_tokens, d_cond)  — CLIP has 77 max
out = cross_attn(latent_feats, text_tokens)
print(f"Cross-attention input:  {latent_feats.shape}")
print(f"Text context:           {text_tokens.shape}")
print(f"Cross-attention output: {out.shape}")

In [ ]:
# ── Full LDM Training Step ──

def ldm_training_step(encoder, decoder, diffusion_unet, schedule, x0, condition,
                       lambda_kl=1e-6):
    """
    Stage 1 (VAE) + Stage 2 (Latent Diffusion) combined training step.
    In practice these two stages are trained separately.
    
    x0:        (B, 3, H, W) pixel images
    condition: (B, L, d_cond) text or class conditioning tokens
    """
    # Stage 1: encode to latent
    z0, kl = encoder(x0)
    # z0: (B, 4, H/8, W/8)

    # Stage 2: diffusion in latent space
    B = z0.shape[0]
    t = torch.randint(0, schedule.T, (B,))
    zt, noise = schedule.q_sample(z0.detach(), t)
    
    # U-Net denoising (simplified: just use a linear layer for demo)
    noise_pred = torch.randn_like(zt)  # placeholder for actual U-Net

    L_diffusion = F.mse_loss(noise_pred, noise)
    
    # VAE reconstruction loss
    x_hat = decoder(z0)
    L_recon = F.mse_loss(x_hat, x0)
    
    # Total loss (in practice, stages are separate)
    L_vae   = L_recon + lambda_kl * kl
    L_total = L_diffusion + L_vae
    return L_total, L_diffusion.item(), L_recon.item(), kl.item()


# ── Stable Diffusion with diffusers library (inference demo) ──
# Uncomment and run if you have diffusers installed:
#
# from diffusers import StableDiffusionPipeline
# pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5",
#                                                 torch_dtype=torch.float16)
# pipe = pipe.to("cuda")
# image = pipe("a photograph of an astronaut riding a horse",
#              num_inference_steps=50, guidance_scale=7.5).images[0]
# image.save("output.png")
print("LDM demo: for inference use `diffusers` library (see comment above)")

# Compression ratio visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

# Draw pipeline
boxes = [
    (0.0,  "Input Image\n512×512×3\n(786K dims)", "#4CAF50"),
    (0.25, "VAE Encode\n(f=8)", "#2196F3"),
    (0.45, "Latent z\n64×64×4\n(16K dims)", "#FF9800"),
    (0.65, "Diffusion\nU-Net (LDM)", "#9C27B0"),
    (0.82, "VAE Decode", "#2196F3"),
]
for x_pos, label, color in boxes:
    ax.text(x_pos + 0.07, 0.5, label, ha='center', va='center',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.6))
    if x_pos < 0.82:
        ax.annotate('', xy=(x_pos + 0.19, 0.5), xytext=(x_pos + 0.14, 0.5),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.text(0.5, 0.1, 'Compression: 786K → 16K dims  (49× reduction in diffusion compute)',
        ha='center', transform=ax.transAxes, fontsize=10, color='gray')
plt.title("LDM: Diffusion in Latent Space", fontsize=13, pad=10)
plt.tight_layout(); plt.show()

## Results

| Method | FID-50K | Resolution | Speed |
|--------|---------|-----------|-------|
| DALL-E | 27.5 | 256² | — |
| VQGAN + Transformer | 15.78 | 256² | slow |
| ADM-G | 4.59 | 256² | ~5 hrs (A100) |
| **LDM-4** | **3.60** | **256²** | **~1.5 days (A100)** |
| **LDM-8 (SD)** | **5.11** | **256²** | **~1.5 days (A100, 4× cheaper)** |

Stable Diffusion (LDM-8 on LAION-5B) is the production implementation with `f=8`.

## Summary

| Component | Description |
|-----------|-------------|
| **VAE** | Compress image to latent (f=8 downsampling), regularise with KL |
| **U-Net** | Denoising backbone operating on 64×64×4 latents |
| **Cross-attention** | Injects text/image/class conditioning into U-Net |
| **Classifier-Free Guidance** | `ε̃ = ε(z_t) + s·(ε(z_t, y) − ε(z_t))` at inference |
| **Efficiency** | 49× fewer dimensions → dramatically cheaper than pixel-space diffusion |

### Key Equation

**Latent diffusion objective:**
$$L_{\text{LDM}} = \mathbb{E}_{z \sim E(x), \epsilon, t}\left[\|\epsilon - \epsilon_\theta(z_t, t, \tau_\theta(y))\|^2\right]$$